In [1]:
!pip install kagglehub

In [2]:
from pathlib import Path
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    # 1) Look for local file first: ../raw_data/dataset.csv
    base_dir = Path.cwd().parent          # one level up from notebook
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    # 2) If not found, download from Kaggle once via kagglehub
    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",  # main CSV name on Kaggle
    )

    # Save for future offline use
    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

df = load_cashflow_data()

Loading local file: /home/diogo/code/cf_copilot/raw_data/dataset.csv


In [3]:
df.head()


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1.930438e+09,2020-01-26,20200125,20200126,20200210.0,USD,RV,1.0,NaN,54273.28,20200126.0,NAH4,1.930438e+09,0
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1.929646e+09,2019-07-22,20190722,20190722,20190811.0,USD,RV,1.0,NaN,79656.60,20190722.0,NAD1,1.929646e+09,0
2,U001,0200792734,MDV/ trust,2019-12-30 00:00:00,2019.0,1.929874e+09,2019-09-14,20190914,20190914,20190929.0,USD,RV,1.0,NaN,2253.86,20190914.0,NAA8,1.929874e+09,0
3,CA02,0140105686,SYSC llc,NaN,2020.0,2.960623e+09,2020-03-30,20200330,20200330,20200410.0,CAD,RV,1.0,NaN,3299.70,20200331.0,CA10,2.960623e+09,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25 00:00:00,2019.0,1.930148e+09,2019-11-13,20191113,20191113,20191128.0,USD,RV,1.0,NaN,33133.29,20191113.0,NAH4,1.930148e+09,0


# Date parsing 
Three formats exist in this dataset:

lets make them all the same format


In [4]:
# Date parsing 

def parse_int_date(series):
    return pd.to_datetime(
        series.astype(str).str.replace('.0', '', regex=False),
        format='%Y%m%d', errors='coerce'
    )

df['clear_date']           = pd.to_datetime(df['clear_date'], errors='coerce')
df['posting_date']         = pd.to_datetime(df['posting_date'], errors='coerce')
df['document_create_date'] = parse_int_date(df['document_create_date'])
df['due_in_date']          = parse_int_date(df['due_in_date'])
df['baseline_create_date'] = parse_int_date(df['baseline_create_date'])
df["document_create_date.1"] = parse_int_date(df["document_create_date.1"])
                                              
#Fix column names
df.rename(columns={
    'buisness_year': 'business_year',
    'document type': 'document_type',
}, inplace=True)

df['business_year'] = df['business_year'].astype(int)
df['doc_id']        = df['doc_id'].astype(str)

print('Parsed. isOpen distribution:')
print(df['isOpen'].value_counts())

Parsed. isOpen distribution:
isOpen
0    40000
1    10000
Name: count, dtype: int64


In [5]:
df.head()

,business_code,cust_number,name_customer,clear_date,business_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document_type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11,2020,1930438491.0,2020-01-26,2020-01-25,2020-01-26,2020-02-10,USD,RV,1.0,NaN,54273.28,2020-01-26,NAH4,1.930438e+09,0
1,U001,0200980828,BEN E,2019-08-08,2019,1929646410.0,2019-07-22,2019-07-22,2019-07-22,2019-08-11,USD,RV,1.0,NaN,79656.60,2019-07-22,NAD1,1.929646e+09,0
2,U001,0200792734,MDV/ trust,2019-12-30,2019,1929873765.0,2019-09-14,2019-09-14,2019-09-14,2019-09-29,USD,RV,1.0,NaN,2253.86,2019-09-14,NAA8,1.929874e+09,0
3,CA02,0140105686,SYSC llc,NaT,2020,2960623488.0,2020-03-30,2020-03-30,2020-03-30,2020-04-10,CAD,RV,1.0,NaN,3299.70,2020-03-31,CA10,2.960623e+09,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25,2019,1930147974.0,2019-11-13,2019-11-13,2019-11-13,2019-11-28,USD,RV,1.0,NaN,33133.29,2019-11-13,NAH4,1.930148e+09,0


# Drop only truly useless columns — name_customer is KEPT as an output identifier

In [8]:
# Drop only truly useless columns — name_customer is KEPT as an output identifier
COLS_TO_DROP = [
    "area_business",          # 100% null — confirmed
    "posting_id",             # constant = 1.0, zero variance
    "document_create_date.1", # exact duplicate of document_create_date
]
df.drop(columns=COLS_TO_DROP, inplace=True, errors="ignore")

# Deduplication
n_before = len(df)
df.drop_duplicates(subset=["doc_id"], keep="last", inplace=True)
print(f"Removed {n_before - len(df)} duplicate rows.")

# Date sanity: due must be >= baseline_create_date
bad = df["due_in_date"] < df["baseline_create_date"]
df = df[~bad].copy()
print(f"Removed {bad.sum()} rows where due_in_date < baseline_create_date.")

print(f"\nFinal shape: {df.shape}")
print(f"  Closed (train):  {(df['isOpen']==0).sum():,}")
print(f"  Open (predict):  {(df['isOpen']==1).sum():,}")

Removed 0 duplicate rows.
Removed 0 rows where due_in_date < baseline_create_date.

Final shape: (48839, 16)
  Closed (train):  39,158
  Open (predict):  9,681


In [9]:
df.head()

,business_code,cust_number,name_customer,clear_date,business_year,doc_id,posting_date,document_create_date,due_in_date,invoice_currency,document_type,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11,2020,1930438491.0,2020-01-26,2020-01-25,2020-02-10,USD,RV,54273.28,2020-01-26,NAH4,1.930438e+09,0
1,U001,0200980828,BEN E,2019-08-08,2019,1929646410.0,2019-07-22,2019-07-22,2019-08-11,USD,RV,79656.60,2019-07-22,NAD1,1.929646e+09,0
2,U001,0200792734,MDV/ trust,2019-12-30,2019,1929873765.0,2019-09-14,2019-09-14,2019-09-29,USD,RV,2253.86,2019-09-14,NAA8,1.929874e+09,0
3,CA02,0140105686,SYSC llc,NaT,2020,2960623488.0,2020-03-30,2020-03-30,2020-04-10,CAD,RV,3299.70,2020-03-31,CA10,2.960623e+09,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25,2019,1930147974.0,2019-11-13,2019-11-13,2019-11-28,USD,RV,33133.29,2019-11-13,NAH4,1.930148e+09,0
